# 04 - Inference, control and CPU benchmarking

Everything here runs on CPU, which is the deployment target. Use this notebook
to measure real-time factor, to inspect and correct pronunciations, and to
export a deployable bundle.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"

# ---------------------------------------------------------------------------
# PICK YOUR EXPERIMENT HERE. This is the only line to change.
#
#   configs/exp0_small.yaml     2013 clips, ~2 GB   -> proves the pipeline,
#                                                     runs on a 6 GB GPU
#   configs/exp1_egyptian.yaml  15.6k clips, 68 h   -> the real run
# ---------------------------------------------------------------------------
CONFIG = "configs/exp0_small.yaml"

from adaptts.utils.config import load_config
from adaptts.utils.logging_utils import setup_logging
setup_logging()
cfg = load_config(CONFIG)
print("repo   :", REPO)
# Printed because the CATT env exists alongside this one: notebook 01 calls the
# CATT interpreter for one stage, and a shell that activated it would otherwise
# hijack a bare `python`. Every stage below runs {sys.executable}, so it follows
# the kernel rather than the shell.
print("python :", sys.executable)
print("config :", CONFIG, "->", cfg.name)
print("dataset:", cfg.paths.hf_dataset_id)

# The homographs named in the brief, read from the probe file so the notebooks
# never hardcode a word list of their own.
import json as _json
PROBE_WORDS = sorted({
    w for _s in _json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]
    for w in _s["focus"].split(" / ") if w and w != "none"
})
print("probe  :", " ".join(PROBE_WORDS))

In [ ]:
import torch
torch.set_num_threads(os.cpu_count() or 4)
from adaptts.infer.pipeline import AdapTTS, save_wav

tts = AdapTTS.from_checkpoints(CONFIG, device="cpu")
print("loaded on CPU with", torch.get_num_threads(), "threads")

## 1. Inspect: what does the model think each word means?

`analyze` returns a plan. It lists every ambiguous word, the reading chosen, the
confidence, and the difficulty that drives adaptive depth.

In [ ]:
plan = tts.analyze(
    "انا كنت مصر على ان مصر عندها امكانيات و موارد تخليها تتفوق على دول من اللي شايفين نفسهم دول"
)
print(plan)

## 2. Correct: override a reading without diacritics

The pronunciation decision is a discrete code, so changing it is a legal edit
that the acoustic model was trained to consume.

In [ ]:
from IPython.display import Audio, display

# The model decides on its own from context.
plan = tts.analyze("انا شوفت علم مصر بيرفرف")
print(plan)
wav, st = tts.synthesize(plan)
display(Audio(wav, rate=st["sample_rate"]))

### Correcting a reading: write the vowels, not a code number

If a reading is wrong, add diacritics to that one word. Nothing else changes, and
the rest of the sentence stays undiacritized.

```
انا كنت مصر على ان مصر عندها امكانيات          # model decides both
انا كنت مُصِرّ على ان مصر عندها امكانيات        # first pinned, second free
```

Why not a code number? Codes are assigned by corpus frequency, so there is no way
to know that code 1 means مُصِرّ, and the answer changes whenever the lexicon is
rebuilt. Writing the vowel is the notation people already use.

**The diacritics never reach the model.** They are read as instructions, resolved
to a code, and stripped, so the model's input is identical either way. A partial
marking is enough: only the letters that distinguish the readings matter.

In [ ]:
free   = "انا كنت مصر على ان مصر عندها امكانيات"
pinned = "انا كنت مُصِرّ على ان مصر عندها امكانيات"

for label, txt in [("model decides", free), ("first word pinned", pinned)]:
    plan = tts.analyze(txt)
    print(f"--- {label} ---")
    print("  text the model sees:", plan.text)
    for w in plan.hard_words:
        src = "user" if w.from_user else "model"
        print(f"    #{w.index} {w.word}: code {w.code} ({src}), "
              f"confidence {w.confidence:.3f}")
    if plan.unresolved_marks:
        print("  could not apply:", plan.unresolved_marks)
    print()

In [ ]:
# Listen to the difference the correction makes.
for label, txt in [("model's choice", free), ("corrected", pinned)]:
    plan = tts.analyze(txt)
    wav, st = tts.synthesize(plan)
    print(label)
    display(Audio(wav, rate=st["sample_rate"]))

In [ ]:
# A correction that cannot be resolved is reported, never guessed at.
for attempt in ["مُصِرّ", "مصِر", "مُصر", "مصُر"]:
    code_, reason = tts.lexicon.code_from_partial_marks(attempt)
    verdict = f"code {code_}" if code_ >= 0 else "unresolved"
    print(f"  {attempt:<8} -> {verdict:<12} {reason}")

### Phoneme variants

Some letters have a second realisation that the diacritizer marks and the model
learns. Type `~` after the letter to force it:

| Written | Effect |
|---|---|
| `ق~` | qaf pronounced as a glottal stop, as in دلوق~تي |
| `ج~` | geem as /zh/ rather than /g/, as in تكنولوج~يا |
| `ف~` | faa as /v/, as in ف~يديو |
| `...ةْ` | sukun on a final ة sounds the /t/: مَدِينَةْ is /madiinat/, مَدِينَة is /madiina/ |

These are independent of the reading: a word can take a variant without being
ambiguous at all.

In [ ]:
for txt in ["دلوقتي احنا مشغولين", "دلوق~تي احنا مشغولين",
            "المدينة كبيره", "المدينةْ كبيره"]:
    plan = tts.analyze(txt)
    v = plan.inputs["variants"][0].tolist()
    marked = [(i, c) for i, c in enumerate(v) if c]
    print(f"{txt:<26} variants at {marked}")

## 2b. The complexity view

Where the compute goes, per word. This is the adaptive claim made checkable: a
word with one reading should be cheap, a homograph should not.

If single-reading words cost as much as homographs, the difficulty head has
learned nothing and the adaptive-depth story is empty.

In [ ]:
print(tts.analyze(
    "انا كنت مصر على ان مصر عندها امكانيات و موارد تخليها تتفوق على دول"
).complexity_report())

In [ ]:
# Easy sentence against hard sentence: the depth should differ.
for txt in ["الجو النهارده حلو", free]:
    plan = tts.analyze(txt)
    n_amb = len(plan.hard_words)
    print(f"difficulty {plan.sentence_difficulty:.3f} -> depth {plan.depth}  "
          f"({n_amb} ambiguous)  {txt[:44]}")

## 2c. The other controls

| Control | What it does |
|---|---|
| `set_tempo(x)` | speaking rate; 1.0 is the model's own pace |
| `set_cfg_scale(x)` | how strongly to follow the conditioning / reference voice |
| `set_budget(f)` | compute ceiling as a fraction of the deepest exit |
| `set_depth(n)` | fix the depth, overriding adaptivity |
| `set_temperature(x)` | sampling randomness |

A budget is a ceiling, not a target: an easy sentence still runs shallow, so it
trades quality for speed only where the model wanted the compute.

In [ ]:
plan = tts.analyze(free)
print("adaptive depth:", plan.depth, "of", list(cfg.acoustic.exit_layers))

for t in [0.8, 1.0, 1.3]:
    p = tts.analyze(free).set_tempo(t)
    wav, st = tts.synthesize(p)
    print(f"tempo {t}: {st['audio_seconds']:.2f}s audio, "
          f"RTF {st['total_seconds']/st['audio_seconds']:.3f}")
    display(Audio(wav, rate=st["sample_rate"]))

In [ ]:
# A budget lowers the depth; it can never raise it.
for b in [0.4, 0.7, 1.0]:
    p = tts.analyze(free).set_budget(b)
    print(f"budget {b} -> depth {p.depth}")

## 3. Benchmark on CPU

Real-time factor below 1.0 means faster than real time. Pocket TTS reports about
6x real time on an M4; expect a similar order here at the shallow exits.

In [ ]:
import time
import numpy as np

sentences = [
    "الجو النهارده حلو جدا",
    "احنا رايحين السوق بكرة الصبح ان شاء الله",
    "انا كنت مصر على ان مصر عندها امكانيات تخليها تتفوق على دول",
]

rows = []
for txt in sentences:
    plan = tts.analyze(txt)
    tts.synthesize(plan)                       # warm up
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        wav, st = tts.synthesize(plan)
        times.append(time.perf_counter() - t0)
    rows.append((len(txt), plan.sentence_difficulty, st["depth"],
                 st["audio_seconds"], float(np.median(times))))

print(f"{'chars':>6}{'difficulty':>12}{'depth':>7}{'audio_s':>9}{'wall_s':>8}{'RTF':>7}")
print("-" * 49)
for n, d, dep, a, t in rows:
    print(f"{n:>6}{d:>12.3f}{dep:>7}{a:>9.2f}{t:>8.2f}{t / a:>7.2f}")

In [ ]:
# Adaptive versus fixed depth, on the same sentences.
import numpy as np

def bench(txt, depth=None):
    plan = tts.analyze(txt)
    if depth:
        plan.set_depth(depth)
    tts.synthesize(plan)
    ts = []
    for _ in range(3):
        t0 = time.perf_counter()
        _, st = tts.synthesize(plan)
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts)), st["depth"]

full = cfg.acoustic.exit_layers[-1]
print(f"{'sentence':<34}{'adaptive':>12}{'fixed-full':>12}{'saving':>9}")
print("-" * 67)
for txt in sentences:
    ta, da = bench(txt)
    tf, _ = bench(txt, full)
    print(f"{txt[:32]:<34}{ta:>10.2f}s{tf:>10.2f}s{(1 - ta / tf) * 100:>8.0f}%")

## 4. Save audio to disk

In [ ]:
wav, st = tts.tts("اهلا بيكم في التجربة الاولى من النظام الجديد")
save_wav("runs/exp1/samples/demo.wav", wav, st["sample_rate"])
print("wrote runs/exp1/samples/demo.wav", st)
display(Audio(wav, rate=st["sample_rate"]))

## 5. Export a deployment bundle

Collects the two checkpoints, the vocabulary, the discovered codes and the
config into one directory that can be copied to a device.

In [ ]:
import shutil, json
from pathlib import Path

out = Path("runs/exp1/deploy")
out.mkdir(parents=True, exist_ok=True)
for src, dst in [
    (Path(cfg.paths.ckpt_dir) / "context_encoder" / "best.pt", "context_encoder.pt"),
    (Path(cfg.paths.ckpt_dir) / "acoustic" / "best.pt", "acoustic.pt"),
    (Path(cfg.paths.charvocab_path), "char_vocab.json"),
    # The reading lexicon, not the retired clustering one. Inference needs it to
    # know how many readings a word has and what each code means.
    (Path(cfg.paths.reading_lexicon_path), "reading_lexicon.json"),
    (Path("configs/exp1_egyptian.yaml"), "config.yaml"),
]:
    if Path(src).exists():
        shutil.copy(src, out / dst)
        print("copied", dst, f"{Path(src).stat().st_size / 1e6:.1f} MB")
    else:
        print("MISSING", src)
print()
print("bundle at", out.resolve())